In [4]:
# ============================================================
# TECH CHALLENGE - FASE 3
# Assistente Médico com RAG + LangChain + LangGraph
# ============================================================

# Instale antes de rodar (no terminal):
# pip install langchain langchain-anthropic langchain-community
# pip install faiss-cpu sentence-transformers
# pip install datasets pandas numpy
# pip install langgraph
# pip install unsloth  # para fine-tuning

import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

print("✅ Imports OK")

✅ Imports OK


In [8]:
# PubMedQA: perguntas e respostas baseadas em artigos do PubMed
# É um dos datasets mais usados para treino de LLMs médicas

from datasets import load_dataset

print("⏳ Baixando PubMedQA...")
dataset_pubmed = load_dataset("qiaojin/PubMedQA", "pqa_labeled")

print(f"✅ PubMedQA carregado!")
print(f"   Split train: {len(dataset_pubmed['train'])} exemplos")
print(f"\n📋 Exemplo de registro:")
exemplo = dataset_pubmed['train'][0]
print(f"   Pergunta : {exemplo['question'][:150]}...")
print(f"   Resposta : {exemplo['long_answer'][:150]}...")
print(f"   Label    : {exemplo['final_decision']}")

⏳ Baixando PubMedQA...
✅ PubMedQA carregado!
   Split train: 1000 exemplos

📋 Exemplo de registro:
   Pergunta : Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?...
   Resposta : Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other or...
   Label    : yes


In [10]:
# MedQuAD: 47.457 pares de perguntas e respostas médicas
# Fonte: NIH, CDC, FDA e outras instituições de saúde

print("⏳ Baixando MedQuAD...")
dataset_medquad = load_dataset("lavita/MedQuAD")

print(f"✅ MedQuAD carregado!")
print(f"   Total de exemplos: {len(dataset_medquad['train'])}")
print(f"\n📋 Exemplo de registro:")
ex = dataset_medquad['train'][0]
print(f"   Pergunta : {str(ex['question'])[:150]}...")
print(f"   Resposta : {str(ex['answer'])[:150]}...")

⏳ Baixando MedQuAD...
✅ MedQuAD carregado!
   Total de exemplos: 47441

📋 Exemplo de registro:
   Pergunta : What is (are) keratoderma with woolly hair ?...
   Resposta : Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-th...


In [11]:
# Prepara os dados para dois usos:
# 1. Fine-tuning (formato instrução)
# 2. RAG (documentos indexados no vector store)

def preprocessar_pubmedqa(dataset, max_exemplos=500):
    """
    Converte PubMedQA para formato de instrução médica.
    Aplica limpeza e anonimização básica.
    """
    registros = []

    for i, item in enumerate(dataset['train']):
        if i >= max_exemplos:
            break

        pergunta = item['question'].strip()
        resposta = item['long_answer'].strip()
        decisao  = item['final_decision']

        # Filtra respostas muito curtas (baixa qualidade)
        if len(resposta) < 50:
            continue

        # Formato de instrução para fine-tuning
        instrucao = {
            "instruction": (
                "Você é um assistente médico especializado. "
                "Responda a pergunta clínica com base em evidências científicas."
            ),
            "input":  pergunta,
            "output": f"{resposta} [Conclusão: {decisao}]",
            "source": "PubMedQA"
        }
        registros.append(instrucao)

    return registros


def preprocessar_medquad(dataset, max_exemplos=500):
    """
    Converte MedQuAD para formato de instrução médica.
    """
    registros = []

    for i, item in enumerate(dataset['train']):
        if i >= max_exemplos:
            break

        pergunta = str(item.get('question', '')).strip()
        resposta = str(item.get('answer',   '')).strip()

        if len(pergunta) < 10 or len(resposta) < 50:
            continue
        if resposta.lower() == 'none' or not resposta:
            continue

        instrucao = {
            "instruction": (
                "Você é um assistente médico especializado. "
                "Responda a pergunta de saúde de forma clara e precisa."
            ),
            "input":  pergunta,
            "output": resposta,
            "source": "MedQuAD"
        }
        registros.append(instrucao)

    return registros


# Processa os dois datasets
print("⏳ Processando PubMedQA...")
dados_pubmed = preprocessar_pubmedqa(dataset_pubmed)

print("⏳ Processando MedQuAD...")
dados_medquad = preprocessar_medquad(dataset_medquad)

# Une os dois
dados_completos = dados_pubmed + dados_medquad
print(f"\n✅ Total de exemplos processados: {len(dados_completos)}")
print(f"   PubMedQA : {len(dados_pubmed)}")
print(f"   MedQuAD  : {len(dados_medquad)}")

⏳ Processando PubMedQA...
⏳ Processando MedQuAD...

✅ Total de exemplos processados: 1000
   PubMedQA : 500
   MedQuAD  : 500


In [12]:
# Analisa a qualidade e distribuição dos dados processados
df = pd.DataFrame(dados_completos)

print("=== ESTATÍSTICAS DOS DADOS ===\n")
print(f"Total de exemplos: {len(df)}")
print(f"Fontes: {df['source'].value_counts().to_dict()}")

# Comprimento das perguntas e respostas
df['len_pergunta'] = df['input'].str.len()
df['len_resposta'] = df['output'].str.len()

print(f"\nComprimento médio das perguntas : {df['len_pergunta'].mean():.0f} caracteres")
print(f"Comprimento médio das respostas : {df['len_resposta'].mean():.0f} caracteres")
print(f"Resposta mais curta  : {df['len_resposta'].min()} caracteres")
print(f"Resposta mais longa  : {df['len_resposta'].max()} caracteres")

print(f"\n📋 Exemplo de dado processado:")
print(json.dumps(df.iloc[0].to_dict(), indent=2, ensure_ascii=False)[:500])

=== ESTATÍSTICAS DOS DADOS ===

Total de exemplos: 1000
Fontes: {'PubMedQA': 500, 'MedQuAD': 500}

Comprimento médio das perguntas : 74 caracteres
Comprimento médio das respostas : 590 caracteres
Resposta mais curta  : 55 caracteres
Resposta mais longa  : 4626 caracteres

📋 Exemplo de dado processado:
{
  "instruction": "Você é um assistente médico especializado. Responda a pergunta clínica com base em evidências científicas.",
  "input": "Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?",
  "output": "Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best of our knowledge, this is the first report of mitochondr


In [13]:
# Salva em dois formatos:
# 1. JSONL para fine-tuning
# 2. CSV para RAG

Path("../data/processed").mkdir(parents=True, exist_ok=True)

# JSONL para fine-tuning
caminho_jsonl = "../data/processed/dados_medicos_ft.jsonl"
with open(caminho_jsonl, 'w', encoding='utf-8') as f:
    for item in dados_completos:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

# CSV para RAG e análise
caminho_csv = "../data/processed/dados_medicos_rag.csv"
df.to_csv(caminho_csv, index=False, encoding='utf-8')

print("✅ Dados salvos!")
print(f"   Fine-tuning (JSONL) : {caminho_jsonl}")
print(f"   RAG (CSV)          : {caminho_csv}")
print(f"   Total de registros : {len(dados_completos)}")

✅ Dados salvos!
   Fine-tuning (JSONL) : ../data/processed/dados_medicos_ft.jsonl
   RAG (CSV)          : ../data/processed/dados_medicos_rag.csv
   Total de registros : 1000
